In [1]:

from transformers import AutoModel, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import os
from tqdm import tqdm

# ── 1. Quantization + Model (your existing code) ──────────────────────────────
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # add this for speed
)

processor = AutoProcessor.from_pretrained("facebook/ijepa_vitg16_22k")

model = AutoModel.from_pretrained(
    "facebook/ijepa_vitg16_22k",
    quantization_config=quantization_config,
    attn_implementation="sdpa",
    device_map="auto"
)

# ── 2. Data ───────────────────────────────────────────────────────────────────










/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/645 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [2]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


class IJEPAClassifier(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(1408, num_classes)

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        x = outputs.last_hidden_state.mean(dim=1)
        x = x.float()  # ← add this line, cast float16 → float32
        logits = self.classifier(x)
        return logits



trainable params: 5,406,720 || all params: 1,016,775,296 || trainable%: 0.5318


In [3]:
# I-JEPA ViT-G was pretrained on 224×224 images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet stats
                         std=[0.229, 0.224, 0.225]),
])

full_train = CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_ds    = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:19<00:00, 8.96MB/s]


In [4]:

# 90/10 train-val split
val_size   = int(0.1 * len(full_train))
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

In [5]:
# ── 3. Model, optimizer, scheduler ───────────────────────────────────────────
# Don't call .cuda() — device_map="auto" already placed the backbone
# Only the classifier head needs to be moved
model.gradient_checkpointing_enable()
classifier_model = IJEPAClassifier(model)
classifier_model.classifier = classifier_model.classifier.cuda()

# Only optimize LoRA params + classifier head (backbone frozen weights skip automatically)
optimizer = AdamW([
    {"params": classifier_model.classifier.parameters(), "lr": 1e-3},
    {"params": [p for p in classifier_model.backbone.parameters() if p.requires_grad], "lr": 2e-4},
], weight_decay=0.01)

NUM_EPOCHS = 10
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

criterion = nn.CrossEntropyLoss()

In [6]:
# ── 4. Helpers ────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, train=True):
    model.classifier.train() if train else model.classifier.eval()
    # backbone eval always — batchnorm/dropout in frozen layers shouldn't update
    model.backbone.eval()

    total_loss, correct, total = 0, 0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images = images.cuda().half()   # float16 to match compute dtype
            labels = labels.cuda()

            logits = model(pixel_values=images)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], max_norm=1.0
                )
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct    += (logits.argmax(dim=1) == labels).sum().item()
            total      += labels.size(0)

    return total_loss / total, correct / total

# ── 5. Checkpointing ──────────────────────────────────────────────────────────

In [7]:
CKPT_DIR = "/content/drive/MyDrive/ijepa_qlora"
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(epoch, val_acc):
    # save_pretrained only saves LoRA adapter weights (tiny)
    classifier_model.backbone.save_pretrained(f"{CKPT_DIR}/lora_epoch{epoch}")
    torch.save(classifier_model.classifier.state_dict(), f"{CKPT_DIR}/head_epoch{epoch}.pt")
    print(f"  Saved checkpoint at epoch {epoch} (val_acc={val_acc:.4f})")

In [ ]:
# ── 6. Training loop ──────────────────────────────────────────────────────────
from tqdm import trange
best_val_acc = 0.0

for epoch in trange(1, NUM_EPOCHS + 1, desc="Epochs"):

    train_loss, train_acc = run_epoch(classifier_model, train_loader, optimizer, train=True)
    val_loss,   val_acc   = run_epoch(classifier_model, val_loader,   train=False)
    scheduler.step()

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(epoch, val_acc)

# ── 7. Final test evaluation ──────────────────────────────────────────────────
test_loss, test_acc = run_epoch(classifier_model, test_loader, train=False)
print(f"\nTest accuracy: {test_acc:.4f}")

Epochs:   0%|          | 0/10 [02:12<?, ?it/s]


KeyboardInterrupt: 